In [244]:
# Import packages
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options

from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import ElementNotInteractableException

from bs4 import BeautifulSoup

import time, re, requests
import pandas as pd

In [245]:
# Open the browser without showing it on screen
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("window-size=2560,1600")

# Open the Chrome browser
driver = webdriver.Chrome() #(options=opts)

# Navigate to the Project Gutenberg homepage
url = "https://www.gutenberg.org/"
driver.get(url)

# Set the implicit waiting time to 0.5 seconds
driver.implicitly_wait(0.5)
# Set the explicit wwaiting time to 20 seconds
wait = WebDriverWait(driver, 20)

In [246]:
# Find the navigation bar links
nav_bar = driver.find_elements(By.CSS_SELECTOR, ".main-links a")
if nav_bar:
    for link in nav_bar:
        # Print the navigation bar links
        print(link.text, link.get_attribute("href"))

 https://www.gutenberg.org/browse/scores/top
 https://www.gutenberg.org/ebooks/categories
Reading Lists https://www.gutenberg.org/ebooks/bookshelf/
Search Options https://www.gutenberg.org/ebooks/


In [247]:
# Navigate to the Reading Lists
reading_lists = driver.find_element(By.CSS_SELECTOR, "a[class='link-reading-lists']")
reading_lists.click()

# Pause execution for 5 seconds
time.sleep(5)

In [248]:
# Navigate to the Main Categories webpage
main_categories = driver.find_element(By.CSS_SELECTOR, "div[class='page_content'] p a")
main_categories.click()

# Pause execution for 5 seconds
time.sleep(5)

In [249]:
# Navigate to the Science-Fiction & Fantasy webpage
sf_fantasy = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/ul/li[14]/a")
sf_fantasy.click()

time.sleep(5)

In [ ]:
def scrape_rows():
    """
    Scrape the rows from the table of a specific year.

    Returns:
        dict: Scraped table row values.
    """
    # Load Selenium's page_source into BeautifulSoup
    document = BeautifulSoup(driver.page_source, "html.parser")

    book_on_page_index = 2
    title_index = 1
    author_index = 2
    download_index = 3

    titles_xpath = f'//*[@id="content"]/div/ul/li[{book_on_page_index}]/a/span[2]/span[{title_index}]'
    authors_xpath = f'//*[@id="content"]/div/ul/li[{book_on_page_index}]/a/span[2]/span[{author_index}]'
    downloads_xpath = f'//*[@id="content"]/div/ul/li[{book_on_page_index}]/a/span[2]/span[{download_index}]'
    texts_xpath = f'//*[@id="content"]/div/ul/li[{book_on_page_index}]/a'

    titles = []
    authors = []
    downloads = []
    texts = []
    for book in driver.find_elements(By.CSS_SELECTOR, ".booklink"):
        if document.select(".title"):
            print(f'Title: {titles_xpath}')
            titles.append(driver.find_element(By.XPATH, titles_xpath))
            title_index += 1
        else:
            print(f'Title: {titles_xpath}')
            titles.append("NaN")
            title_index += 1
        if document.select(".subtitle"):
            print(f'Author: {authors_xpath}')
            authors.append(driver.find_element(By.XPATH, authors_xpath))
            author_index += 1
        else:
            print(f'Author: {authors_xpath}')
            authors.append("NaN")
            author_index += 1
        if document.select(".extra"):
            print(f'Download: {downloads_xpath}')
            downloads.append(driver.find_element(By.XPATH, downloads_xpath))
            download_index += 1
        else:
            print(f'Download: {downloads_xpath}')
            downloads.append("NaN")
            download_index += 1
        if driver.find_elements(By.CSS_SELECTOR, ".booklink a"):
            print(f'Text: {texts_xpath}')
            texts.append(driver.find_element(By.XPATH, texts_xpath))
        else:
            print(f'Text: {texts_xpath}')
            texts.append("NaN")
        book_on_page_index += 1

    return {
        "titles": titles,
        "authors": authors,
        "downloads": downloads,
        "texts": texts
    }

In [274]:
def create_text():
    """
    Turn the scraped rows into text.

    Returns:
        dict: Table row values in text form.
    """
    # Assign scrape_rows() function to scrape_rows_result variable
    scrape_rows_result = scrape_rows()

    return {
        "title_text": [title.text for title in scrape_rows_result["titles"]],
        "author_text": [author.text for author in scrape_rows_result["authors"]],
        "download_text": [download.text for download in scrape_rows_result["downloads"]],
        "text_text": [text.get_attribute("href") for text in scrape_rows_result["texts"]]
    }

In [275]:
def collect_data():
    """
    Iterates over table rows and scrapes it's content.
    
    Returns:
        List: Non-emtpy table rows.
    """
    # Assign create_text() function to create_text_result variable
    create_text_result = create_text()
    # Count the amount of table rows
    row_count = len(driver.find_elements(By.CSS_SELECTOR, ".booklink"))

    # Append text to list
    specific_page = []
    row_index = 0
    for row in range(row_count):
        specific_page.append({
            "Title": create_text_result["title_text"][row_index],
            "Author": create_text_result["author_text"][row_index],
            "Downloads": create_text_result["download_text"][row_index],
            "Text": create_text_result["text_text"][row_index]
        })
        row_index += 1

    return specific_page

In [276]:
# Instantiate the df_novels DataFrame
df_novels = pd.DataFrame()

# Count the amount of books on the webpage
page_length = 169

# Iterate through the webpages
for page in range(1):
    # Create the df_append DataFrame, based on the results of the scrape_page() function
    df_append = pd.DataFrame(collect_data())
    # Append the df_append DataFrame to the df_novels DataFrame
    df_novels = pd.concat([df_novels, df_append], ignore_index=True)

    try:
        driver.find_element(By.CSS_SELECTOR, "a[title='Go to the next page of results.']").click()
        time.sleep(3)
    except NoSuchElementException:
        pass

Title: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[1]
Author: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[2]
Download: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[3]
Text: //*[@id="content"]/div/ul/li[2]/a
Title: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[1]
Author: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[2]
Download: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[3]
Text: //*[@id="content"]/div/ul/li[2]/a
Title: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[1]
Author: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[2]
Download: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[3]
Text: //*[@id="content"]/div/ul/li[2]/a
Title: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[1]
Author: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[2]
Download: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[3]
Text: //*[@id="content"]/div/ul/li[2]/a
Title: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[1]
Author: //*[@id="content"]/div/ul/li[2]/a/span[2]/span[2]
Download: //*[@id="conten

In [277]:
# Show the df_novels DataFrame
df_novels.head(50)

,Title,Author,Downloads,Text
0,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
1,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
2,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
3,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
4,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
5,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
6,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
7,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
8,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
9,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads,https://www.gutenberg.org/ebooks/84
